In [ ]:
# ============================================================
# ASSIGNMENT 2 - Q1: CLICK-HISTORY & SESSION FEATURES (MIND, LARGE)
# Team: Shivam Patel + Suyash Pande | CS4.406 IRE
#
# Builds ALL behavioural features on MINDLARGE with strict behaviour-window boundary
# (no future-click leakage). GBDT feature matrix on a 600k sample of MINDlarge_train
# (GBDT saturates ~400-600k). Also builds the labeled DEV matrix (ablations in A2_04).
#
# Feature families (Q1):
#  1. Click-history: recency-weighted history (exp decay), click count, hist-embedding sim
#  2. Session: position bias, slate depth (dwell N/A for MIND)
#  3. Article: rolling popularity, freshness, CTR, category match
#  4. Behaviour-window boundary: strictly-before-T counting + Q9 leakage assertion
#
# Saves: mind_large_features.npz  (consumed by A2_02 / A2_03 / A2_04)
# ============================================================
!pip install lightgbm sentence-transformers -q
import os, glob, re, math, numpy as np, pandas as pd, datetime as dt, random
from bisect import bisect_left
from collections import defaultdict, Counter
random.seed(0)

LTR = glob.glob("/kaggle/input/**/MINDlarge_train", recursive=True)[0]
LDV = glob.glob("/kaggle/input/**/MINDlarge_dev",   recursive=True)[0]
print("large train:", LTR)
print("large dev:  ", LDV)

GBDT_SAMPLE = 600_000
DEV_SAMPLE  = 40_000

NEWS = ["news_id","category","subcategory","title","abstract","url","te","ae"]
BEH  = ["impression_id","user_id","time","history","impressions"]
_WORD= re.compile(r"[^\W\d_]+", re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if isinstance(t,str) else []
def pfx(x): return f"mind:{x}"
print("config: GBDT_SAMPLE", GBDT_SAMPLE, "| DEV_SAMPLE", DEV_SAMPLE)


In [ ]:
# ---- article catalogue (train+dev) + MiniLM embeddings ----
news = pd.concat([pd.read_csv(f"{d}/news.tsv", sep="\t", header=None, names=NEWS, quoting=3,
                 usecols=["news_id","category","title","abstract"]) for d in (LTR, LDV)]
                ).drop_duplicates("news_id").reset_index(drop=True)
news["title"]=news["title"].fillna(""); news["abstract"]=news["abstract"].fillna("")
ids=[pfx(r.news_id) for r in news.itertuples()]
id_to_row={x:i for i,x in enumerate(ids)}
cat_lut={pfx(r.news_id):(r.category if isinstance(r.category,str) else "") for r in news.itertuples()}

from sentence_transformers import SentenceTransformer
minilm=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb=minilm.encode([f"{r.title} {r.abstract}".strip() for r in news.itertuples()],
                  batch_size=512, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
emb_by_id={ids[i]:emb[i] for i in range(len(ids))}
print("catalogue:", len(ids), "| embeddings:", emb.shape)


In [ ]:
# ---- behaviour logs + point-in-time event index (BEHAVIOUR-WINDOW BOUNDARY) ----
def load_beh(p):
    b=pd.read_csv(f"{p}/behaviors.tsv",sep="\t",header=None,names=BEH,quoting=3)
    b["t"]=pd.to_datetime(b["time"],format="%m/%d/%Y %I:%M:%S %p",errors="coerce"); return b
print("loading large train behaviors (big read)...")
b_tr=load_beh(LTR); b_dv=load_beh(LDV)
print("train impressions:", len(b_tr), "| dev impressions:", len(b_dv))

hist_lut={}
for b in (b_tr,b_dv):
    for u,h in zip(b["user_id"],b["history"]):
        if isinstance(h,str) and h: hist_lut[pfx(u)]=[pfx(x) for x in h.split()]

click_ev=defaultdict(list); imp_ev=defaultdict(list); first_seen={}
for b in (b_tr,b_dv):
    for t,imps in zip(b["t"],b["impressions"]):
        if pd.isna(t) or not isinstance(imps,str): continue
        for tkn in imps.split():
            p=tkn.split("-"); nid=pfx(p[0])
            if nid not in first_seen or t<first_seen[nid]: first_seen[nid]=t
            if len(p)==2:
                imp_ev[nid].append(t)
                if p[1]=="1": click_ev[nid].append(t)
for d in (click_ev,imp_ev):
    for k in d: d[k].sort()
print("users w/ history:", len(hist_lut), "| articles w/ clicks:", len(click_ev))


In [ ]:
# ---- FEATURE FUNCTIONS (all point-in-time: only events strictly before T) ----
def cnt(d,aid,T,w=None):
    tl=d.get(aid)
    if not tl: return 0
    hi=bisect_left(tl,T); return hi if w is None else hi-bisect_left(tl,T-dt.timedelta(hours=w))
def ctr(aid,T,w=None):
    c=cnt(click_ev,aid,T,w); s=cnt(imp_ev,aid,T,w); return c/s if s>0 else 0.0
def freshness(aid,T,tau=6.0):
    fs=first_seen.get(aid)
    if fs is None or T is None: return 0.0
    dh=(T-fs).total_seconds()/3600.0
    return float(np.exp(-dh/tau)) if dh>=0 else 0.0
def recency_wt_hist(uid,mh=50,tau=8.0):
    ai=hist_lut.get(uid,[])[-mh:]
    vs=[emb_by_id[a] for a in ai if a in emb_by_id]
    if not vs: return None
    n=len(vs); w=np.exp(-(np.arange(n)[::-1])/tau); w/=w.sum()
    um=np.average(np.array(vs),axis=0,weights=w); return um/(np.linalg.norm(um)+1e-9)
def user_cat_dist(uid,mh=50):
    ai=hist_lut.get(uid,[])[-mh:]; cats=[cat_lut.get(a) for a in ai]
    tot=len([c for c in cats if c]); cc=Counter(c for c in cats if c)
    return {k:v/tot for k,v in cc.items()} if tot else {}
def mm(x):
    lo,hi=x.min(),x.max(); return np.zeros_like(x) if hi-lo<1e-12 else (x-lo)/(hi-lo)
print("feature functions ready (leakage-safe click-history / session / article)")


In [ ]:
# ---- Q9 LEAKAGE ASSERTION on large train ----
violations=0; checked=0; seen=0
for row in b_tr.itertuples():
    if pd.isna(row.t) or not isinstance(row.impressions,str): continue
    for tkn in row.impressions.split():
        c=pfx(tkn.split("-")[0]); tl=click_ev.get(c,[]); k=cnt(click_ev,c,row.t)
        violations += sum(1 for x in tl[:k] if x>=row.t); checked+=1
    seen+=1
    if seen>=5000: break
print(f"checked {checked} pairs | future-click leakage violations: {violations}")
assert violations==0, "LEAKAGE DETECTED"
print("OK: behaviour-window boundary enforced (Q9 passed).")


In [ ]:
# ---- build feature matrices: 600k train sample + labeled dev ----
FEAT = ["hist_emb_mean","hist_emb_best","recency_wt_sim","click_count",
        "pop_1h","pop_24h","pop_7d","pop_total","ctr_24h","ctr_total",
        "freshness","cat_match","position","slate_size"]

def collect_rows(b_df):
    rows=[]
    for iid,u,t,imps in zip(b_df["impression_id"],b_df["user_id"],b_df["t"],b_df["impressions"]):
        if not isinstance(imps,str) or pd.isna(t): continue
        cand=[];labs=[]
        for tkn in imps.split():
            p=tkn.split("-")
            if len(p)==2: cand.append(pfx(p[0])); labs.append(int(p[1]))
        if cand and sum(labs)>0: rows.append((iid,pfx(u),t,cand,np.array(labs)))
    return rows

def build(rows):
    X=[];Y=[];G=[];META=[];IIDS=[]
    for iid,uid,T,cand,labs in rows:
        hv=[emb_by_id[a] for a in hist_lut.get(uid,[])[-50:] if a in emb_by_id]
        um=(np.mean(hv,0)/(np.linalg.norm(np.mean(hv,0))+1e-9)) if hv else None
        rw=recency_wt_hist(uid); uc=user_cat_dist(uid)
        hlen=len(hist_lut.get(uid,[])); m=len(cand); F=[]
        for i,c in enumerate(cand):
            cv=emb_by_id.get(c)
            em=float(um@cv) if (um is not None and cv is not None) else 0.0
            eb=float(max((v@cv for v in hv),default=0.0)) if cv is not None else 0.0
            rws=float(rw@cv) if (rw is not None and cv is not None) else 0.0
            p1=cnt(click_ev,c,T,1);p24=cnt(click_ev,c,T,24);p7=cnt(click_ev,c,T,168);ptot=cnt(click_ev,c,T)
            c24=ctr(c,T,24);ctot=ctr(c,T)
            fr=freshness(c,T);cm=uc.get(cat_lut.get(c,""),0.0);pos=i/max(1,m-1)
            F.append([em,eb,rws,hlen,p1,p24,p7,ptot,c24,ctot,fr,cm,pos,m])
        F=np.array(F,float)
        for col in [4,5,6,7]: F[:,col]=mm(F[:,col])
        X.append(F);Y.append(labs);G.append(m);META.append((uid,T,cand));IIDS.append(iid)
    return np.vstack(X),np.concatenate(Y),G,META,IIDS

print("collecting train rows...")
tr_rows=collect_rows(b_tr)
random.Random(0).shuffle(tr_rows); tr_rows=tr_rows[:GBDT_SAMPLE]
print(f"building features for {len(tr_rows)} train impressions (slow step)...")
Xtr,ytr,gtr,_,iids_tr=build(tr_rows)

print("collecting dev rows...")
dv_rows=collect_rows(b_dv)
random.Random(0).shuffle(dv_rows); dv_rows=dv_rows[:DEV_SAMPLE]
print(f"building features for {len(dv_rows)} dev impressions...")
Xdv,ydv,gdv,meta_dv,iids_dv=build(dv_rows)

print("train:", Xtr.shape, "| dev:", Xdv.shape)
np.savez_compressed("/kaggle/working/mind_large_features.npz",
                    Xtr=Xtr,ytr=ytr,gtr=np.array(gtr),Xdv=Xdv,ydv=ydv,gdv=np.array(gdv),
                    feat=FEAT, iids_dv=np.array(iids_dv), iids_tr=np.array(iids_tr))
print("saved /kaggle/working/mind_large_features.npz")
print("features:", FEAT)
# push updated npz (now with impression IDs) to HF
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi
    _tok=UserSecretsClient().get_secret("HF_TOKEN")
    HfApi(token=_tok).upload_file(path_or_fileobj="/kaggle/working/mind_large_features.npz",
        path_in_repo="mind_large_features.npz", repo_id="donbosoc/mind-artifacts", repo_type="dataset")
    print("pushed updated npz (with iids) to HF")
except Exception as e:
    print("HF push skipped/failed:", e)
